# Machine Unlearning — Colab Runner
### Master's Research: Machine Unlearning for Multi-Class Image Classification
**Author:** Mikołaj Hajder 264478

This notebook is the **entry point for all cloud experiments**.  
It handles:
1. Mounting Google Drive (checkpoints persist across sessions)
2. Cloning / pulling the latest code from GitHub
3. Installing dependencies
4. Running training and unlearning scripts via shell cells

> **Workflow:** run the setup cells **once per session**, then jump straight to the experiment cell you need.


## 1. Mount Google Drive
Checkpoints, logs, and data are stored here so nothing is lost when the Colab session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
GDRIVE = '/content/drive/MyDrive/master_thesis'
CKPT_DIR = f'{GDRIVE}/checkpoints'
DATA_DIR = f'{GDRIVE}/data'

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print(f'Checkpoints → {CKPT_DIR}')
print(f'Data        → {DATA_DIR}')


## 2. Clone / Update the Repository

In [ ]:
import os

REPO_URL = 'https://github.com/<YOUR_USERNAME>/master_thesis.git'  # ← update this
REPO_DIR = '/content/master_thesis'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
!git log --oneline -3


## 3. Install Dependencies

In [ ]:
%cd /content/master_thesis
!pip install -q -r requirements.txt
print('Dependencies installed.')


## 4. Verify Setup

In [ ]:
%cd /content/master_thesis
!python -c "
import torch, torchvision, yaml
from models import build_resnet18
from utils import STATS, evaluate, set_seed
print(f'PyTorch     : {torch.__version__}')
print(f'Torchvision : {torchvision.__version__}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
print('All imports OK.')
"


---
## 5. Run Experiments

Each cell below runs one experiment. They are **independent** — you can run them in any order after setup.

> **Tip:** The `--checkpoint-dir` flag saves all `.pth` files directly to Google Drive,  
> so if the session dies mid-training, the last saved checkpoint is safe.


### 5a. Train — CIFAR-10

In [ ]:
%cd /content/master_thesis
!python train.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5b. Train — CIFAR-100

In [ ]:
%cd /content/master_thesis
!python train.py \
    --config configs/cifar100.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5c. Naive Unlearning — CIFAR-10 (random forget, 1% of train set)

In [ ]:
%cd /content/master_thesis
!python unlearn_naive.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR}


### 5d. Naive Unlearning — CIFAR-10 (class-wise forget, e.g. class 0 = 'airplane')

In [ ]:
%cd /content/master_thesis
!python unlearn_naive.py \
    --config configs/cifar10.yaml \
    --checkpoint-dir {CKPT_DIR} \
    --data-root {DATA_DIR} \
    --forget-strategy class \
    --forget-class 0


---
## 6. Download Checkpoints (optional)

If you want to download a checkpoint to your local machine for analysis:


In [ ]:
from google.colab import files
import os

# List available checkpoints
ckpts = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pth')]
print('Available checkpoints:')
for c in sorted(ckpts):
    size_mb = os.path.getsize(os.path.join(CKPT_DIR, c)) / 1e6
    print(f'  {c}  ({size_mb:.1f} MB)')


In [ ]:
# Uncomment the line below and set the filename to download a specific checkpoint
# files.download(os.path.join(CKPT_DIR, 'resnet18_cifar10_best.pth'))
